# Diabetes Dataset Preprocessing

This notebook prepares the Pima Indians Diabetes dataset for a Deep Learning task by:
1. Replacing biologically impossible zeros with column medians
2. Removing outliers using the IQR method
3. Applying StandardScaler for feature normalization
4. Exporting the cleaned data to `diabetes_cleaned.csv`

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

# ── Load Dataset ──────────────────────────────────────────────────────
df = pd.read_csv('../dataset/diabetes.csv')
print(f'Original shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
df.head()

Original shape: (768, 9)
Columns: ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age', 'Outcome']


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


## Step 1 — Handle Biological Impossibilities

Columns like Glucose, BloodPressure, SkinThickness, Insulin, and BMI **cannot be zero** in a living patient.  
We replace those zeros with the **median** of each respective column (robust to outliers).

In [2]:
# Columns where 0 is biologically impossible
zero_invalid_cols = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']

# Count zeros before replacement
print('Zero counts BEFORE replacement:')
for col in zero_invalid_cols:
    n_zeros = (df[col] == 0).sum()
    print(f'  {col:20s}: {n_zeros} zeros')

# Replace 0 → NaN → median
df[zero_invalid_cols] = df[zero_invalid_cols].replace(0, np.nan)
for col in zero_invalid_cols:
    median_val = df[col].median()
    df[col].fillna(median_val, inplace=True)

print('\nZero counts AFTER replacement:')
for col in zero_invalid_cols:
    n_zeros = (df[col] == 0).sum()
    print(f'  {col:20s}: {n_zeros} zeros')

Zero counts BEFORE replacement:
  Glucose             : 5 zeros
  BloodPressure       : 35 zeros
  SkinThickness       : 227 zeros
  Insulin             : 374 zeros
  BMI                 : 11 zeros

Zero counts AFTER replacement:
  Glucose             : 0 zeros
  BloodPressure       : 0 zeros
  SkinThickness       : 0 zeros
  Insulin             : 0 zeros
  BMI                 : 0 zeros


C:\Users\yusuf\AppData\Local\Temp\ipykernel_21576\3296044251.py:14: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  df[col].fillna(median_val, inplace=True)
C:\Users\yusuf\AppData\Local\Temp\ipykernel_21576\3296044251.py:14: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using

## Step 2 — Outlier Removal (IQR Method)

For every **feature column**, we compute Q1 and Q3 and remove any row where a feature falls outside \[Q1 − 1.5 × IQR,  Q3 + 1.5 × IQR\].

In [3]:
feature_cols = df.columns.drop('Outcome')
shape_before_iqr = df.shape

Q1 = df[feature_cols].quantile(0.25)
Q3 = df[feature_cols].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

# Keep only rows where ALL features are within bounds
mask = ((df[feature_cols] >= lower_bound) & (df[feature_cols] <= upper_bound)).all(axis=1)
df = df[mask].reset_index(drop=True)

print(f'Shape before IQR removal: {shape_before_iqr}')
print(f'Shape after  IQR removal: {df.shape}')
print(f'Rows removed: {shape_before_iqr[0] - df.shape[0]}')

Shape before IQR removal: (768, 9)
Shape after  IQR removal: (338, 9)
Rows removed: 430


## Step 3 — Feature Scaling (StandardScaler)

Scale all features to **mean ≈ 0** and **std ≈ 1** so the neural network converges faster.

In [4]:
scaler = StandardScaler()
df[feature_cols] = scaler.fit_transform(df[feature_cols])

print('After StandardScaler (should be ~0 mean, ~1 std):')
print(df[feature_cols].describe().loc[['mean', 'std']].round(4))

After StandardScaler (should be ~0 mean, ~1 std):
      Pregnancies  Glucose  BloodPressure  SkinThickness  Insulin     BMI  \
mean      -0.0000   0.0000         0.0000         0.0000   0.0000  0.0000   
std        1.0015   1.0015         1.0015         1.0015   1.0015  1.0015   

      DiabetesPedigreeFunction     Age  
mean                    0.0000 -0.0000  
std                     1.0015  1.0015  


## Step 4 — Export Cleaned Dataset

In [5]:
output_path = '../dataset/diabetes_cleaned.csv'
df.to_csv(output_path, index=False)
print(f'Saved cleaned dataset to: {output_path}')

Saved cleaned dataset to: ../dataset/diabetes_cleaned.csv


## Step 5 — Verification Summary

In [6]:
print('=' * 55)
print('           PREPROCESSING SUMMARY')
print('=' * 55)
print(f'  Original dataset shape : (768, 9)')
print(f'  After cleaning shape   : {df.shape}')
print(f'  Rows removed           : {768 - df.shape[0]}')
print(f'  Features scaled        : {list(feature_cols)}')
print(f'  Target column          : Outcome (unchanged)')
print(f'  Output file            : {output_path}')
print('=' * 55)
print()
print('Class distribution after cleaning:')
print(df['Outcome'].value_counts().to_string())
print()
df.head()

           PREPROCESSING SUMMARY
  Original dataset shape : (768, 9)
  After cleaning shape   : (338, 9)
  Rows removed           : 430
  Features scaled        : ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age']
  Target column          : Outcome (unchanged)
  Output file            : ../dataset/diabetes_cleaned.csv

Class distribution after cleaning:
Outcome
0    240
1     98



,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,-0.734222,-1.053106,-0.421732,-0.532393,-0.524790,-0.681177,-1.223164,-0.975968,0
1,-0.070674,-1.429411,-1.876278,0.352118,-0.605631,-0.221557,-0.901042,-0.453349,1
2,0.592874,1.581026,0.123722,-0.925508,0.566566,-1.045703,0.447097,2.159747,1
3,-1.065996,-0.061031,1.214632,1.826302,1.307609,2.124089,0.303932,0.069270,1
4,-0.734222,-0.163659,-0.058096,0.155560,-0.497843,0.349005,0.216442,0.173794,1
